# NB28 · Testing, Debugging & Logging

`advanced/` · 20 questions · the last one is a story project

How professionals trust their code: `assert` checks, `unittest`, pytest-style
test functions, `doctest`, mocks that stand in for slow or random things,
reading tracebacks, and `logging` instead of scattered `print()` calls. Every
task is a **write** task. Test runners are told to stay quiet and the tasks
ask you to print only the stable facts (passed or failed, counts, names).

## You'll need

- NB23: decorators
- NB24: context managers

## You'll practise

- assert-based tests; arrange-act-assert; choosing edge cases
- unittest: TestCase, assertions, setUp, running inside a notebook
- pytest style: plain test functions, fixtures and parametrize (concepts)
- doctest
- unittest.mock: Mock, patch, side_effect, assert_called_with
- Debugging: reading tracebacks, the traceback module, breakpoint()/pdb
- logging: levels, getLogger, handlers, formatters; warnings

## How it works

Write your code in the cell under each question, run it, and compare with **Expected output**. Open **Solution** only after a real attempt.

Questions get longer as you go: one-liners first, a small story project last. Every solution only uses what this notebook and the ones before it teach.

---

## Warm-up · one line each

### Q01 · Real-life

`slugify` turns a title into a URL piece. Pin it down with three `assert`
statements: a plain title, a title with extra spaces around it, and a title
that is already lower case. Then print `slugify ok`.

**Expected output:**

```text
slugify ok
```

<details><summary><b>Solution</b></summary>

```python
assert slugify("Hello World") == "hello-world"
assert slugify("  Big   News ") == "big-news"
assert slugify("python") == "python"
print("slugify ok")
```

</details>

In [ ]:
def slugify(title):
    return "-".join(title.lower().split())

# your code here


### Q02 · Output-match

The VAT helper below should add 20%. Check `add_vat(10)` against `12` with
an `assert` that has the message `VAT should be 20%`, wrapped so that the
failure is caught and only the message is printed.

**Expected output:**

```text
VAT should be 20%
```

<details><summary><b>Solution</b></summary>

```python
try:
    assert add_vat(10) == 12, "VAT should be 20%"
except AssertionError as error:
    print(error)
```

</details>

In [ ]:
def add_vat(price):
    return price * 1.1

# your code here


### Q03 · Output-match

A teammate left a `breakpoint()` call in `checkout()`, which would freeze a
script waiting for a debugger. Without editing `checkout`, make
`breakpoint()` do nothing by setting the `PYTHONBREAKPOINT` environment
variable to `0` from Python, then call `checkout()`.

**Expected output:**

```text
checkout finished
```

<details><summary><b>Solution</b></summary>

```python
import os
os.environ["PYTHONBREAKPOINT"] = "0"
checkout()
```

</details>

In [ ]:
def checkout():
    breakpoint()
    print("checkout finished")

# your code here


### Q04 · Puzzle

`is_adult` is supposed to be `True` from age 18 upwards. One carefully
chosen `assert` exposes the bug. Write it with a `try`/`except` so that
instead of crashing the program prints `bug found at the boundary`.

**Expected output:**

```text
bug found at the boundary
```

<details><summary><b>Solution</b></summary>

```python
try:
    assert is_adult(18) is True
except AssertionError:
    print("bug found at the boundary")
```

</details>

In [ ]:
def is_adult(age):
    return age > 18

# your code here


### Q05 · Real-life

`mailer` stands in for a real email service. Create it as a `Mock`, call
`mailer.send("ana@example.com", "Welcome")` once, then print how many times
`send` was called and the arguments it received (as a tuple).

**Expected output:**

```text
1
('ana@example.com', 'Welcome')
```

<details><summary><b>Solution</b></summary>

```python
from unittest.mock import Mock
mailer = Mock()
mailer.send("ana@example.com", "Welcome")
print(mailer.send.call_count)
print(mailer.send.call_args.args)
```

</details>

In [ ]:
# your code here


---

## Combine · 2 to 4 lines

### Q06 · Real-life

Configure logging so everything goes to standard output with the format
`LEVEL:message` (use `%(levelname)s:%(message)s`) and the level `INFO`.
Then log one `debug`, one `info` and one `warning` message with any text.
Only the last two should appear.

**Expected output:**

```text
INFO:server starting
WARNING:disk almost full
```

<details><summary><b>Solution</b></summary>

```python
import logging, sys
logging.basicConfig(stream=sys.stdout, force=True, level=logging.INFO,
                    format="%(levelname)s:%(message)s")
logging.debug("loading config")
logging.info("server starting")
logging.warning("disk almost full")
```

</details>

In [ ]:
# your code here


### Q07 · Build

Write a `unittest.TestCase` called `ConvertTests` with two tests for
`c_to_f`: one for `0` (32.0) and one for `100` (212.0). Run it with a
`TextTestRunner` whose stream is a `StringIO`, then print whether the run
was successful and how many tests ran, like the expected output.

**Expected output:**

```text
successful: True
tests run: 2
```

<details><summary><b>Solution</b></summary>

```python
import io, unittest
class ConvertTests(unittest.TestCase):
    def test_freezing(self):
        self.assertEqual(c_to_f(0), 32.0)
    def test_boiling(self):
        self.assertEqual(c_to_f(100), 212.0)
suite = unittest.defaultTestLoader.loadTestsFromTestCase(ConvertTests)
result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print("successful:", result.wasSuccessful())
print("tests run:", result.testsRun)
```

</details>

In [ ]:
def c_to_f(c):
    return c * 9 / 5 + 32

# your code here


### Q08 · Real-life

`weather` is a fake weather service whose `get()` fails once with a
`TimeoutError` and then succeeds. Loop twice and call `weather.get()`
inside a `try`; print `timeout, retrying` on a failure and the data when it
works.

**Expected output:**

```text
timeout, retrying
sunny, 21C
```

<details><summary><b>Solution</b></summary>

```python
for attempt in range(2):
    try:
        print(weather.get())
    except TimeoutError:
        print("timeout, retrying")
```

</details>

In [ ]:
from unittest.mock import Mock
weather = Mock()
weather.get.side_effect = [TimeoutError("slow"), "sunny, 21C"]

# your code here


### Q09 · Puzzle

`ratio` calls `average`, which can divide by zero. Call `ratio([])` inside
a `try`, catch the error and use the `traceback` module on
`error.__traceback__` to print the names of the functions the error passed
through, outermost first, joined with ` -> `.

**Expected output:**

```text
<module> -> ratio -> average
```

<details><summary><b>Solution</b></summary>

```python
import traceback
try:
    ratio([])
except ZeroDivisionError as error:
    frames = traceback.extract_tb(error.__traceback__)
    print(" -> ".join(frame.name for frame in frames))
```

</details>

In [ ]:
def average(values):
    return sum(values) / len(values)

def ratio(values):
    return average(values) * 100

# your code here


### Q10 · Build

The docstring of `double` contains two examples. Use `doctest` to run them
without printing anything yourself from the examples: build the test with
`doctest.DocTestParser().get_doctest(double.__doc__, {"double": double},
"double", None, 0)`, run it with a `doctest.DocTestRunner()` and print the
number of attempted examples and the number of failed ones.

**Expected output:**

```text
attempted: 2
failed: 0
```

<details><summary><b>Solution</b></summary>

```python
import doctest
test = doctest.DocTestParser().get_doctest(double.__doc__, {"double": double}, "double", None, 0)
result = doctest.DocTestRunner().run(test)
print("attempted:", result.attempted)
print("failed:", result.failed)
```

</details>

In [ ]:
def double(n):
    """
    >>> double(4)
    8
    >>> double("ab")
    'abab'
    """
    return n * 2

# your code here


---

## Small real tasks · 5 to 8 lines

### Q11 · Real-life

Build a logger hierarchy for a shop. Create the loggers `shop` and
`shop.cart`. Give **only** `shop` a stream handler (to `sys.stdout`) with
the format `%(name)s %(levelname)s %(message)s`, and set its level to
`WARNING`. Then make `shop.cart` log one `info` and one `warning` message.
The child has no handler of its own, so its record travels up to the parent.

**Expected output:**

```text
shop.cart WARNING stock is low
```

<details><summary><b>Solution</b></summary>

```python
import logging, sys
shop = logging.getLogger("shop")
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(name)s %(levelname)s %(message)s"))
shop.addHandler(handler)
shop.setLevel(logging.WARNING)
cart = logging.getLogger("shop.cart")
cart.info("item added")
cart.warning("stock is low")
```

</details>

In [ ]:
# your code here


### Q12 · Build

Write a `CartTests` test case for the `Cart` class. Use `setUp` to create a
fresh cart with `add("pen", 2)` already done. Write three tests: the total
is 2 (`assertEqual`), adding a negative quantity raises `ValueError`
(`assertRaises`), and removing something absent raises `KeyError`. Run the
suite quietly and print `successful`, `failures` and `errors` counts.

**Expected output:**

```text
successful: True
failures: 0
errors: 0
```

<details><summary><b>Solution</b></summary>

```python
import io, unittest
class CartTests(unittest.TestCase):
    def setUp(self):
        self.cart = Cart()
        self.cart.add("pen", 2)
    def test_total(self):
        self.assertEqual(self.cart.total(), 2)
    def test_negative_quantity(self):
        with self.assertRaises(ValueError):
            self.cart.add("pen", -1)
    def test_remove_missing(self):
        with self.assertRaises(KeyError):
            self.cart.remove("book")
suite = unittest.defaultTestLoader.loadTestsFromTestCase(CartTests)
result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print("successful:", result.wasSuccessful())
print("failures:", len(result.failures))
print("errors:", len(result.errors))
```

</details>

In [ ]:
class Cart:
    def __init__(self):
        self.items = {}
    def add(self, name, qty):
        if qty < 0:
            raise ValueError('negative quantity')
        self.items[name] = self.items.get(name, 0) + qty
    def remove(self, name):
        del self.items[name]
    def total(self):
        return sum(self.items.values())

# your code here


### Q13 · Real-life

`describe(city)` calls `fetch_json`, which would hit the internet. Use
`unittest.mock.patch` on **this module's** `fetch_json` (the path is
`"__main__.fetch_json"`) with the return value `{"temp": 18}`, call
`describe("Oslo")` while patched and print the result. Afterwards print
what `patch` recorded: the single argument `fetch_json` was called with.

**Expected output:**

```text
Oslo: 18 C
https://weather.example/Oslo
```

<details><summary><b>Solution</b></summary>

```python
from unittest.mock import patch
with patch("__main__.fetch_json", return_value={"temp": 18}) as fake:
    print(describe("Oslo"))
print(fake.call_args.args[0])
```

</details>

In [ ]:
def fetch_json(url):
    raise RuntimeError("no internet in tests")

def describe(city):
    data = fetch_json("https://weather.example/" + city)
    return city + ": " + str(data["temp"]) + " C"

# your code here


### Q14 · Real-life

`Session` expires `ttl` seconds after it was created, based on a clock
function passed in. Test it without waiting: create a `Mock` clock with
`side_effect=[100, 105, 130]` (the first reading is used when the session is
created), build a `Session` with `ttl=20` using it, then print
`is_expired()` twice. The two calls read the clock at 105 and 130.

**Expected output:**

```text
False
True
```

<details><summary><b>Solution</b></summary>

```python
from unittest.mock import Mock
clock = Mock(side_effect=[100, 105, 130])
session = Session(20, clock)
print(session.is_expired())
print(session.is_expired())
```

</details>

In [ ]:
class Session:
    def __init__(self, ttl, clock):
        self.ttl = ttl
        self.clock = clock
        self.created = clock()
    def is_expired(self):
        return self.clock() - self.created > self.ttl

# your code here


### Q15 · Build

pytest runs plain `test_` functions, injects fixtures and repeats a test
for many cases (`parametrize`). Imitate that by hand. Write a function
`make_cart()` (the fixture) returning a fresh `Cart`, and a function
`check_total(cart, items, expected)` (the test) that adds the `items`
`(name, qty)` pairs and asserts the total. Run it for every case in
`cases`, each with a fresh cart, and print `PASS case-number` or
`FAIL case-number` (numbering from 1) using `try`/`except AssertionError`.
The given `Cart.add` has a bug (adding the same item twice overwrites
instead of adding up), so one case is expected to FAIL: your runner should
report it, not hide it.

**Expected output:**

```text
PASS 1
PASS 2
FAIL 3
```

<details><summary><b>Solution</b></summary>

```python
def make_cart():
    return Cart()

def check_total(cart, items, expected):
    for name, qty in items:
        cart.add(name, qty)
    assert cart.total() == expected

for number, (items, expected) in enumerate(cases, start=1):
    try:
        check_total(make_cart(), items, expected)
        print("PASS", number)
    except AssertionError:
        print("FAIL", number)
```

</details>

In [ ]:
class Cart:
    def __init__(self):
        self.items = {}
    def add(self, name, qty):
        self.items[name] = qty
    def total(self):
        return sum(self.items.values())

cases = [([("pen", 2)], 2), ([("pen", 2), ("ink", 3)], 5), ([("pen", 2), ("pen", 3)], 5)]

# your code here


---

## Puzzles & builds · 8 to 15 lines

### Q16 · Puzzle

`median` has bugs. Write a helper `report(label, ok)` that prints
`PASS label` or `FAIL label`. Use it for four checks of the buggy function:
`median([3])` is 3, `median([1, 3, 2])` is 2, `median([4, 1, 3, 2])` is 2.5,
and `median([5, 1, 9])` is 5. Then write a corrected `median` (it must sort,
and average the two middle values for an even count) and print a line
`--- after the fix ---` followed by the same four checks.

**Expected output:**

```text
PASS single
FAIL odd unsorted
FAIL even
FAIL another odd
--- after the fix ---
PASS single
PASS odd unsorted
PASS even
PASS another odd
```

<details><summary><b>Solution</b></summary>

```python
def report(label, ok):
    print("PASS" if ok else "FAIL", label)

def run_checks():
    report("single", median([3]) == 3)
    report("odd unsorted", median([1, 3, 2]) == 2)
    report("even", median([4, 1, 3, 2]) == 2.5)
    report("another odd", median([5, 1, 9]) == 5)

run_checks()

def median(values):
    ordered = sorted(values)
    middle = len(ordered) // 2
    if len(ordered) % 2 == 1:
        return ordered[middle]
    return (ordered[middle - 1] + ordered[middle]) / 2

print("--- after the fix ---")
run_checks()
```

</details>

In [ ]:
def median(values):
    middle = len(values) // 2
    return values[middle]

# your code here


### Q17 · Real-life

A payment API is flaky. Write a decorator `retry(times)` that calls the
function up to `times` times. When a `ConnectionError` happens it logs a
`warning` through the logger `payments` with the text
`attempt 1 failed: down` (attempt number and error text) and tries again;
after the last failed attempt the error is raised. Logs must go to a
`StringIO` through a handler with the format `%(levelname)s %(message)s`.
Decorate a function `charge()` that calls the mock `api` and prints the
result, then print everything the log collected.

**Expected output:**

```text
paid
WARNING attempt 1 failed: down
WARNING attempt 2 failed: down
```

<details><summary><b>Solution</b></summary>

```python
import io, logging
stream = io.StringIO()
handler = logging.StreamHandler(stream)
handler.setFormatter(logging.Formatter("%(levelname)s %(message)s"))
log = logging.getLogger("payments")
log.addHandler(handler)
log.setLevel(logging.INFO)
log.propagate = False

def retry(times):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except ConnectionError as error:
                    log.warning("attempt %d failed: %s", attempt, error)
                    if attempt == times:
                        raise
        return wrapper
    return decorator

@retry(3)
def charge():
    return api()

print(charge())
print(stream.getvalue().strip())
```

</details>

In [ ]:
from unittest.mock import Mock
api = Mock(side_effect=[ConnectionError("down"), ConnectionError("down"), "paid"])

# your code here


### Q18 · Puzzle

A plain `Mock` accepts **any** call, so a typo in a test passes for the
wrong reason. Create `lazy = Mock()` and call `lazy.sned("x")` (typo) and
print `plain mock accepted the typo`. Then create `strict =
Mock(spec=Mailer)`, try the same typo inside a `try`/`except
AttributeError` and print `spec mock rejected the typo`. Finally call
`strict.send("x")` and assert with `assert_called_once_with("x")`, printing
`send verified`.

**Expected output:**

```text
plain mock accepted the typo
spec mock rejected the typo
send verified
```

<details><summary><b>Solution</b></summary>

```python
from unittest.mock import Mock
lazy = Mock()
lazy.sned("x")
print("plain mock accepted the typo")

strict = Mock(spec=Mailer)
try:
    strict.sned("x")
except AttributeError:
    print("spec mock rejected the typo")
strict.send("x")
strict.send.assert_called_once_with("x")
print("send verified")
```

</details>

In [ ]:
class Mailer:
    def send(self, text):
        print('really sending', text)

# your code here


### Q19 · Build

Write a `ListHandler` (a `logging.Handler` subclass whose `emit` stores each
record in `self.records`) and attach it to the logger `audit` at level
`INFO`, with propagation off. The function `old_total` is deprecated: it
should emit a `DeprecationWarning` with the text `use new_total` through
`warnings.warn`. Call it inside `warnings.catch_warnings(record=True)` with
`warnings.simplefilter("always")`, then log `"total computed"` at `info`
and `"deprecated call seen"` at `warning`. Print the warning's category
name and message, then every collected record as `LEVEL: message`.

**Expected output:**

```text
DeprecationWarning use new_total
INFO: total computed
WARNING: deprecated call seen
```

<details><summary><b>Solution</b></summary>

```python
import logging, warnings

class ListHandler(logging.Handler):
    def __init__(self):
        super().__init__()
        self.records = []
    def emit(self, record):
        self.records.append(record)

audit = logging.getLogger("audit")
collector = ListHandler()
audit.addHandler(collector)
audit.setLevel(logging.INFO)
audit.propagate = False

def old_total(prices):
    warnings.warn("use new_total", DeprecationWarning)
    return sum(prices)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    total = old_total([2, 3])
audit.info("total computed")
audit.warning("deprecated call seen")
print(caught[0].category.__name__, str(caught[0].message))
for record in collector.records:
    print(record.levelname + ": " + record.getMessage())
```

</details>

In [ ]:
def old_total(prices):
    return sum(prices)

# your code here


---

## Q20 · Story project: Bug Hunt at Pocket Bank

A teammate wrote the banking module in the first cell below, then went on
holiday. Customers already complain. You'll write tests that **expose** the
bugs, fix them, prove the fixes, add logging and test the SMS alert
without sending a real message. Each step builds on the one before.

**Run these cells first.** They set up the starting files and data for every step below.

In [ ]:
class InsufficientFunds(Exception):
    pass


class Account:
    def __init__(self, owner, balance=0, history=[]):
        self.owner = owner
        self.balance = balance
        self.history = history

    def deposit(self, amount):
        self.balance += amount
        self.history.append(("deposit", amount))

    def withdraw(self, amount):
        if amount >= self.balance:
            raise InsufficientFunds(self.owner)
        self.balance -= amount
        self.history.append(("withdraw", amount))

    def apply_interest(self, rate):
        self.balance = self.balance + self.balance * rate


def send_sms(owner, text):
    raise RuntimeError("no SMS gateway in tests")


def notify_low_balance(account, limit=50):
    if account.balance < limit:
        send_sms(account.owner, "Low balance: " + str(account.balance))
        return True
    return False


### Step 1

Write `BankTests` (a `unittest.TestCase`) with four tests that each
expose one problem: (1) depositing a negative amount should raise
`ValueError`; (2) withdrawing exactly the whole balance of 100 should
work and leave 0; (3) two accounts created without a history must not
share one (deposit into one, the other's history stays empty); (4)
`apply_interest(0.1)` on a balance of 33.33 should give 36.66
(rounded to 2 decimals). Write a function `run(test_class)` that runs
a test class quietly and returns the result. Call it and print
`successful`, the sorted names of the failed or errored tests
(`test.id().split(".")[-1]`), and the counts.

**Expected output:**

```text
successful: False
failing: ['test_interest_rounded', 'test_negative_deposit', 'test_separate_histories', 'test_withdraw_everything']
tests run: 4 problems: 4
```

<details><summary><b>Solution</b></summary>

```python
import io, unittest

class BankTests(unittest.TestCase):
    def test_negative_deposit(self):
        account = Account("ana", 10)
        with self.assertRaises(ValueError):
            account.deposit(-5)
    def test_withdraw_everything(self):
        account = Account("ana", 100)
        account.withdraw(100)
        self.assertEqual(account.balance, 0)
    def test_separate_histories(self):
        first = Account("ana")
        second = Account("bo")
        first.deposit(10)
        self.assertEqual(second.history, [])
    def test_interest_rounded(self):
        account = Account("ana", 33.33)
        account.apply_interest(0.1)
        self.assertEqual(account.balance, 36.66)

def run(test_class):
    suite = unittest.defaultTestLoader.loadTestsFromTestCase(test_class)
    return unittest.TextTestRunner(stream=io.StringIO()).run(suite)

result = run(BankTests)
print("successful:", result.wasSuccessful())
bad = sorted(test.id().split('.')[-1] for test, _ in result.failures + result.errors)
print("failing:", bad)
print("tests run:", result.testsRun, "problems:", len(bad))
```

</details>

In [ ]:
# your code here


### Step 2

Fix the module by writing a corrected `Account` class (same
interface). It must reject non-positive deposits with `ValueError`, allow
withdrawing up to the full balance (raise `InsufficientFunds` only
when the amount is **more** than the balance), give every account its
own list (`history=None` default), and round the balance to 2 decimals
after interest. Re-run `BankTests` with `run` and print `successful`
and the number of problems.

**Expected output:**

```text
successful: True
problems: 0
```

<details><summary><b>Solution</b></summary>

```python
class Account:
    def __init__(self, owner, balance=0, history=None):
        self.owner = owner
        self.balance = balance
        self.history = [] if history is None else history

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self.balance += amount
        self.history.append(("deposit", amount))

    def withdraw(self, amount):
        if amount > self.balance:
            raise InsufficientFunds(self.owner)
        self.balance -= amount
        self.history.append(("withdraw", amount))

    def apply_interest(self, rate):
        self.balance = round(self.balance + self.balance * rate, 2)

result = run(BankTests)
print("successful:", result.wasSuccessful())
print("problems:", len(result.failures) + len(result.errors))
```

</details>

In [ ]:
# your code here


### Step 3

Add logging. Write `transfer(source, target, amount)` that withdraws
from `source` and deposits into `target`. Use a logger called
`bank.transfer` with a handler writing to `sys.stdout` (format
`%(levelname)s %(name)s: %(message)s`, level `INFO`, propagation
off). A good transfer logs an `info` line `ana -> bo: 30`; when
`InsufficientFunds` is raised it logs a `warning` line
`ana cannot send 500` and returns `False` (it returns `True` on
success). Try one transfer of 30 and one of 500 between accounts with
balances 100 and 20, then print both balances.

**Expected output:**

```text
INFO bank.transfer: ana -> bo: 30
WARNING bank.transfer: ana cannot send 500
balances: 70 50
```

<details><summary><b>Solution</b></summary>

```python
import logging, sys
log = logging.getLogger("bank.transfer")
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(levelname)s %(name)s: %(message)s"))
log.addHandler(handler)
log.setLevel(logging.INFO)
log.propagate = False

def transfer(source, target, amount):
    try:
        source.withdraw(amount)
    except InsufficientFunds:
        log.warning("%s cannot send %s", source.owner, amount)
        return False
    target.deposit(amount)
    log.info("%s -> %s: %s", source.owner, target.owner, amount)
    return True

ana = Account("ana", 100)
bo = Account("bo", 20)
transfer(ana, bo, 30)
transfer(ana, bo, 500)
print("balances:", ana.balance, bo.balance)
```

</details>

In [ ]:
# your code here


### Step 4

Test the SMS alert without sending anything. `notify_low_balance`
calls the module's `send_sms`. Use `unittest.mock.patch` on
`"__main__.send_sms"` to check two situations with `Account("cy",
20)` and `Account("di", 500)`: the low account returns `True` and
the SMS function was called once with `("cy", "Low balance: 20")`; the
rich account returns `False` and the SMS function was never called.
Print the return value and the call information for each.

**Expected output:**

```text
True
('cy', 'Low balance: 20')
1
False
False
```

<details><summary><b>Solution</b></summary>

```python
from unittest.mock import patch
with patch("__main__.send_sms") as fake_sms:
    low = Account("cy", 20)
    print(notify_low_balance(low))
    print(fake_sms.call_args.args)
    print(fake_sms.call_count)
with patch("__main__.send_sms") as fake_sms:
    rich = Account("di", 500)
    print(notify_low_balance(rich))
    print(fake_sms.called)
```

</details>

In [ ]:
# your code here


### Step 5

Finish with a regression suite. Write `AlertTests` (a `TestCase`)
whose `setUp` starts a patcher for `"__main__.send_sms"` (stop it with
`self.addCleanup`), with two tests: a low balance sends exactly one
SMS, a balance of 50 (exactly the limit) sends none. Then load
**both** `BankTests` and `AlertTests` into one suite (use
`unittest.TestSuite` and `loadTestsFromTestCase`), run it quietly and
print `All green: True` style output plus the total number of tests.

**Expected output:**

```text
All green: True
Total tests: 6
```

<details><summary><b>Solution</b></summary>

```python
class AlertTests(unittest.TestCase):
    def setUp(self):
        patcher = patch("__main__.send_sms")
        self.fake_sms = patcher.start()
        self.addCleanup(patcher.stop)
    def test_low_balance_sends_one(self):
        notify_low_balance(Account("cy", 10))
        self.assertEqual(self.fake_sms.call_count, 1)
    def test_limit_sends_none(self):
        notify_low_balance(Account("cy", 50))
        self.fake_sms.assert_not_called()

loader = unittest.defaultTestLoader
suite = unittest.TestSuite()
suite.addTests(loader.loadTestsFromTestCase(BankTests))
suite.addTests(loader.loadTestsFromTestCase(AlertTests))
result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print("All green:", result.wasSuccessful())
print("Total tests:", result.testsRun)
```

</details>

In [ ]:
# your code here


---

## Done

Next up: **NB29 · Performance Memory** in `advanced/`.